# SelfCheckGPT Experiment 1 â€” NLI Inference (Kaggle/Colab GPU)

**Purpose:** Run Gate 5 (smoke test) + Gate 6 (full 38,160-pair inference) on GPU.

**Requirements:**
- GPU accelerator enabled (T4 or P100)
- Internet access enabled (for HuggingFace downloads)

**Outputs saved to `/kaggle/working/`** (download from the Output tab after run):
- `nli_scores.npy` â€” shape (1908, 20) â€” raw per-sentence/per-sample contradiction scores
- `sentence_scores.npy` â€” shape (1908,) â€” final Eq.10 aggregated scores
- `run_metadata.json` â€” versions, device, config

**Frozen scientific config (DO NOT MODIFY):**
- Dataset: `potsawee/wiki_bio_gpt3_hallucination` (evaluation split)
- NLI model: `potsawee/deberta-v3-large-mnli`
- Formula: Eq. 9-10 from Manakul et al., EMNLP 2023
- N = 20 samples per passage

## Cell 1 â€” GPU Check + Install Dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'sentencepiece',
                'protobuf', 'scikit-learn', 'tqdm'], check=True)

import torch, platform, transformers, datasets as ds_lib
import sklearn, numpy as np

print('='*60)
print('ENVIRONMENT CHECK')
print('='*60)
print(f'Python       : {platform.python_version()}')
print(f'PyTorch      : {torch.__version__}')
print(f'Transformers : {transformers.__version__}')
print(f'Datasets     : {ds_lib.__version__}')
print(f'scikit-learn : {sklearn.__version__}')
print(f'NumPy        : {np.__version__}')
print()
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU          : {gpu}')
    print(f'VRAM         : {vram:.1f} GB')
    DEVICE = 'cuda'
    BATCH_SIZE = 32  # safe for T4 (15GB) and P100 (16GB)
    if vram < 14:
        BATCH_SIZE = 16
        print(f'  -> Low VRAM detected, reducing BATCH_SIZE to {BATCH_SIZE}')
else:
    print('WARNING: No GPU detected! Inference will be very slow on CPU.')
    print('Enable GPU: Settings (right panel) > Accelerator > GPU T4 x1')
    DEVICE = 'cpu'
    BATCH_SIZE = 8

print(f'Device       : {DEVICE}')
print(f'Batch size   : {BATCH_SIZE}')
print('='*60)

## Cell 2 â€” Load + Validate Dataset (Gate 3 verification)

In [ ]:
from datasets import load_dataset

DATASET_NAME = 'potsawee/wiki_bio_gpt3_hallucination'
EXPECTED_PASSAGES  = 238
EXPECTED_SENTENCES = 1908
EXPECTED_SAMPLES   = 20
VALID_LABELS = {'accurate', 'minor_inaccurate', 'major_inaccurate'}

print(f'Loading {DATASET_NAME} [evaluation] ...')
raw = load_dataset(DATASET_NAME)['evaluation']
print(f'Raw rows: {len(raw)}')

# Flatten to sentence-level records
records = []
for passage_id, row in enumerate(raw):
    sentences   = row['gpt3_sentences']
    annotations = row['annotation']
    samples     = row['gpt3_text_samples']
    assert len(samples) == EXPECTED_SAMPLES, \
        f'Passage {passage_id}: {len(samples)} samples, expected {EXPECTED_SAMPLES}'
    assert len(sentences) == len(annotations), \
        f'Passage {passage_id}: alignment broken ({len(sentences)} vs {len(annotations)})'
    for sent_idx, (sent, label) in enumerate(zip(sentences, annotations)):
        records.append({
            'passage_id':   passage_id,
            'sentence_idx': sent_idx,
            'sentence':     sent,
            'label':        label,
            'samples':      samples,
        })

from collections import Counter
label_counts = Counter(r['label'] for r in records)

print()
print('-'*50)
print('DATASET VALIDATION')
print('-'*50)
print(f'Passages  : {len(set(r["passage_id"] for r in records))}  (expected {EXPECTED_PASSAGES})')
print(f'Sentences : {len(records)}  (expected {EXPECTED_SENTENCES})')
print(f'Samples   : {EXPECTED_SAMPLES} per passage')
print(f'Labels    : {dict(label_counts)}')
print()

assert len(set(r['passage_id'] for r in records)) == EXPECTED_PASSAGES
assert len(records) == EXPECTED_SENTENCES
assert all(len(r['samples']) == EXPECTED_SAMPLES for r in records)
assert set(label_counts.keys()) == VALID_LABELS

print('Gate 3 verification: PASSED [OK]')
print(f'Total NLI pairs to compute: {len(records)} x {EXPECTED_SAMPLES} = {len(records)*EXPECTED_SAMPLES}')

## Cell 3 â€” Load NLI Model (potsawee/deberta-v3-large-mnli)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = 'potsawee/deberta-v3-large-mnli'
print(f'Loading tokenizer: {MODEL_NAME} ...')
# use_fast=False: DeBERTa-v3 uses SentencePiece; fast tokenizer path
# incorrectly tries TikToken in Transformers 5.x
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
print('Tokenizer loaded.')

print(f'Loading model: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()
print('Model loaded and moved to device.')

print(f'num_labels               : {model.config.num_labels}')

# ---------------------------------------------------------
# CRITICAL DISCOVERY (Item 8 in discrepancy checklist):
# potsawee/deberta-v3-large-mnli is a CUSTOM 2-class model
# where the neutral class is ALREADY REMOVED at the head level.
# Index 0 = Entailment
# Index 1 = Contradiction
# Further, the official implementation expects inputs ordered as:
# tokenizer(hypothesis, premise)
# ---------------------------------------------------------
ENTAIL_IDX = 0
CONTRA_IDX = 1

print(f'Final: ENTAIL_IDX={ENTAIL_IDX}  CONTRA_IDX={CONTRA_IDX}')

if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated(0)/1e9
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'VRAM after model load: {used:.2f} / {total:.1f} GB used')

## Cell 4 â€” Gate 5: Smoke Test (sanity check before full inference)

In [ ]:
# CELL 4: Evaluation and Smoke Test
import torch
import numpy as np

# Eq. 9 implementation
@torch.inference_mode()
def score_batch(premises, hypotheses, max_length=512):
    """
    Eq. 9: score = P(contra) / (P(entail) + P(contra))
    Because potsawee/deberta-v3-large-mnli is a 2-class model
    without the neutral class, this simplifies to P(contra).
    
    CRITICAL: This model expects (hypothesis, premise) ordering.
    CRITICAL: AutoTokenizer drops [CLS]/[SEP] for sentence pairs.
              We build them manually to guarantee correctness.
    """
    input_ids = []
    token_type_ids = []
    attention_mask = []
    
    for prem, hyp in zip(premises, hypotheses):
        # 1. Tokenize separately (no special tokens)
        h_ids = tokenizer(hyp, add_special_tokens=False)['input_ids']
        p_ids = tokenizer(prem, add_special_tokens=False)['input_ids']
        
        # 2. Build sequence: [CLS] hyp [SEP] prem [SEP]
        seq = [1] + h_ids + [2] + p_ids + [2]
        
        # 3. Build token_type_ids: 0 for hyp, 1 for prem
        type_ids = [0]*(len(h_ids)+2) + [1]*(len(p_ids)+1)
        
        # 4. Truncate if necessary (keeping [CLS] and [SEP])
        if len(seq) > max_length:
            seq = seq[:max_length-1] + [2]
            type_ids = type_ids[:max_length]
            
        input_ids.append(seq)
        token_type_ids.append(type_ids)
        attention_mask.append([1]*len(seq))
        
    # 5. Pad manually ([PAD] = 0)
    max_len = max(len(s) for s in input_ids)
    for i in range(len(input_ids)):
        pad_len = max_len - len(input_ids[i])
        input_ids[i] += [0] * pad_len       
        token_type_ids[i] += [0] * pad_len
        attention_mask[i] += [0] * pad_len
        
    inputs = {
        'input_ids': torch.tensor(input_ids).to(DEVICE),
        'token_type_ids': torch.tensor(token_type_ids).to(DEVICE),
        'attention_mask': torch.tensor(attention_mask).to(DEVICE)
    }
    
    logits = model(**inputs).logits
    probs  = torch.softmax(logits, dim=-1)
    
    # Eq. 9
    score = probs[:, CONTRA_IDX]
    return score.cpu().numpy().astype(np.float32)

# Hand-picked pairs: (premise, hypothesis, expected_direction)
smoke_pairs = [
    ('The sky is blue during the day.',
     'The sky is green during the day.',
     'high', 'Direct color contradiction'),
    
    ('Albert Einstein was born in Ulm, Germany.',
     'Albert Einstein was born in Berlin.',
     'high', 'Birthplace contradiction'),
    
    ('The Eiffel Tower is located in Paris, France.',
     'The Eiffel Tower is located in London.',
     'high', 'WikiBio-like hallucinated sentence'),
    
    ('The sky is blue during the day.',
     'The sky has a blue color during daytime.',
     'low', 'Paraphrase entailment'),
    
    ('Barack Obama was the 44th President of the United States.',
     'Barack Obama was a US President.',
     'low', 'Subset entailment'),
    
    ('Water boils at 100 degrees Celsius at 1 atm.',
     'Water turns to gas at 100 degrees Celsius.',
     'low', 'WikiBio-like factual sentence')
]

print('='*60)
print('GATE 5: SMOKE TEST')
print('='*60)
print('#     Score  Expect  Pass?   Description')
print('-'*60)

smoke_premises = [p[0] for p in smoke_pairs]
smoke_hypotheses = [p[1] for p in smoke_pairs]
smoke_scores = score_batch(smoke_premises, smoke_hypotheses, max_length=128)

pass_count = 0
for i, (prem, hyp, expect, desc) in enumerate(smoke_pairs):
    s = smoke_scores[i]
    is_high = s > 0.5
    passed = (expect == 'high' and is_high) or (expect == 'low' and not is_high)
    pass_count += int(passed)
    marker = '[OK]' if passed else '[FAIL]'
    print(f'{i+1:<4} {s:.4f} {expect:>7}  {marker:<7} {desc}')

print('-'*60)
print(f'Mean contradiction score : {np.mean(smoke_scores[:3]):.4f}  (expect > 0.5)')
print(f'Mean entailment score    : {np.mean(smoke_scores[3:]):.4f}  (expect < 0.5)')

direction_correct = np.mean(smoke_scores[:3]) > np.mean(smoke_scores[3:])
print(f'Direction correct        : {direction_correct}')
print('\n')

if pass_count == 6 and direction_correct:
    gate5_pass = True
    print('GATE 5: PASSED -- Inference logic is fully validated.')
else:
    print('GATE 5: FAILED -- DO NOT proceed. Check label indices and premise/hypothesis order.')
    raise RuntimeError('Gate 5 failed. Aborting.')
print('='*60)


## Cell 5 â€” Build All 38,160 Pairs

In [ ]:
# Build flat lists for batched inference
# premise   = sampled passage (s_j)
# hypothesis = target sentence (r_i)
all_premises    = []
all_hypotheses  = []
idx_map         = []  # (sentence_idx, sample_idx) -> flat position

for i, rec in enumerate(records):
    for j, sample in enumerate(rec['samples']):
        all_premises.append(sample)
        all_hypotheses.append(rec['sentence'])
        idx_map.append((i, j))

total_pairs = len(all_premises)
assert total_pairs == EXPECTED_SENTENCES * EXPECTED_SAMPLES == 38160, \
    f'Expected 38160 pairs, got {total_pairs}'

print(f'Total pairs to score : {total_pairs}')
print(f'Batch size           : {BATCH_SIZE}')
print(f'Total batches        : {(total_pairs + BATCH_SIZE - 1) // BATCH_SIZE}')
print(f'Estimated time (T4)  : ~15-25 min')

## Cell 6 â€” Gate 6: Full Batched Inference (38,160 pairs)

In [ ]:
from tqdm.notebook import tqdm
import time

scores_flat = np.zeros(total_pairs, dtype=np.float32)
t0 = time.time()

for start in tqdm(range(0, total_pairs, BATCH_SIZE), desc='NLI inference'):
    end = min(start + BATCH_SIZE, total_pairs)
    batch_scores = score_batch(
        all_premises[start:end],
        all_hypotheses[start:end]
    )
    scores_flat[start:end] = batch_scores

elapsed = time.time() - t0
print(f'\nInference complete in {elapsed/60:.1f} min')
print(f'Pairs per second: {total_pairs/elapsed:.1f}')

# Validation
assert not np.isnan(scores_flat).any(),   'NaN values found in scores!'
assert scores_flat.min() >= 0.0,          f'Min score below 0: {scores_flat.min()}'
assert scores_flat.max() <= 1.0 + 1e-5,  f'Max score above 1: {scores_flat.max()}'
print(f'scores_flat stats: min={scores_flat.min():.4f}  max={scores_flat.max():.4f}  mean={scores_flat.mean():.4f}')

## Cell 7 â€” Gate 6 Validation: Reshape to (1908, 20) Matrix

In [ ]:
# Reshape flat (38160,) -> matrix (1908, 20)
nli_matrix = np.zeros((EXPECTED_SENTENCES, EXPECTED_SAMPLES), dtype=np.float32)
for (i, j), s in zip(idx_map, scores_flat):
    nli_matrix[i, j] = s

print('NLI matrix validation:')
print(f'  Shape  : {nli_matrix.shape}  (expected (1908, 20))')
print(f'  NaN    : {np.isnan(nli_matrix).sum()}  (expected 0)')
print(f'  Min    : {nli_matrix.min():.4f}')
print(f'  Max    : {nli_matrix.max():.4f}')
print(f'  Mean   : {nli_matrix.mean():.4f}')

assert nli_matrix.shape == (EXPECTED_SENTENCES, EXPECTED_SAMPLES)
assert not np.isnan(nli_matrix).any()
assert nli_matrix.min() >= 0.0
assert nli_matrix.max() <= 1.0 + 1e-5
print()
print('GATE 6: PASSED [OK]')

# Eq. 10: sentence-level score = mean over 20 samples
sentence_scores = nli_matrix.mean(axis=1)  # shape (1908,)
print(f'Sentence scores shape : {sentence_scores.shape}  (expected (1908,))')
print(f'Sentence scores mean  : {sentence_scores.mean():.4f}')

## Cell 8 â€” Save Outputs to /kaggle/working/

In [ ]:
import os, json, platform
import sklearn

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)

# Save matrices
nli_path      = os.path.join(OUT_DIR, 'nli_scores.npy')
sent_path     = os.path.join(OUT_DIR, 'sentence_scores.npy')
meta_path     = os.path.join(OUT_DIR, 'run_metadata.json')
smoke_path    = os.path.join(OUT_DIR, 'smoke_test_scores.npy')

np.save(nli_path,   nli_matrix)
np.save(sent_path,  sentence_scores)
np.save(smoke_path, smoke_scores)

# Save metadata
metadata = {
    'model':                'potsawee/deberta-v3-large-mnli',
    'dataset':              'potsawee/wiki_bio_gpt3_hallucination',
    'split':                'evaluation',
    'n_passages':           EXPECTED_PASSAGES,
    'n_sentences':          EXPECTED_SENTENCES,
    'samples_per_passage':  EXPECTED_SAMPLES,
    'total_pairs':          total_pairs,
    'batch_size':           BATCH_SIZE,
    'device':               DEVICE,
    'gpu':                  (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'),
    'nli_matrix_shape':     list(nli_matrix.shape),
    'nli_matrix_min':       float(nli_matrix.min()),
    'nli_matrix_max':       float(nli_matrix.max()),
    'nli_matrix_mean':      float(nli_matrix.mean()),
    'gate5_passed':         bool(gate5_pass),
    'gate6_passed':         True,
    'smoke_test_scores':    smoke_scores.tolist(),
    'torch_version':        torch.__version__,
    'transformers_version': transformers.__version__,
    'datasets_version':     ds_lib.__version__,
    'sklearn_version':      sklearn.__version__,
    'numpy_version':        np.__version__,
    'python_version':       platform.python_version(),
    'platform':             platform.platform(),
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print('Files saved:')
for path in [nli_path, sent_path, smoke_path, meta_path]:
    size = os.path.getsize(path) / 1e6
    print(f'  {os.path.basename(path):<30} {size:.2f} MB')

print()
print('Download instructions:')
print('  Right panel -> Output tab -> find these files -> click download icon')

## Cell 9 â€” Final Summary

In [ ]:
print('='*60)
print('FINAL SUMMARY')
print('='*60)
print(f'Gate 5 (smoke test)   : {"PASSED" if gate5_pass else "FAILED"}')
print(f'Gate 6 (full matrix)  : PASSED')
print()
print(f'NLI matrix shape      : {nli_matrix.shape}')
print(f'Sentence scores shape : {sentence_scores.shape}')
print()
print('Label distribution in records:')
from collections import Counter
lc = Counter(r['label'] for r in records)
for lbl, cnt in sorted(lc.items()):
    print(f'  {lbl:<22}: {cnt}')
print()
nonfact_labels = np.array([0 if r['label']=='accurate' else 1 for r in records])
factual_labels = np.array([1 if r['label']=='accurate' else 0 for r in records])
print(f'nonfact_labels.sum()  : {nonfact_labels.sum()}  (expected 1392)')
print(f'factual_labels.sum()  : {factual_labels.sum()}  (expected 516)')
print()
print('NEXT STEP: Download the 4 files from the Output tab.')
print('Place them in your local project:')
print('  outputs/scores/nli_scores.npy')
print('  outputs/scores/sentence_scores.npy')
print('  outputs/scores/smoke_test_scores.npy')
print('  outputs/metrics/run_metadata.json')
print()
print('Then run locally: python scripts/run_reproduction.py --skip-inference')
print('='*60)